In [1]:
using ScQubitsMimic
using CairoMakie

_scqubitsmimic_example_dir = normpath(joinpath(dirname(pathof(ScQubitsMimic)), "..", "examples"))
if !isdefined(Main, :ScQubitsMimicExampleMakie)
    include(joinpath(_scqubitsmimic_example_dir, "makie_fontsetup.jl"))
end
ScQubitsMimicExampleMakie.setup_makie_font!()


ArgumentError: ArgumentError: Package ScQubitsMimic not found in current path.
- Run `import Pkg; Pkg.add("ScQubitsMimic")` to install the ScQubitsMimic package.

# Tunable Custom Circuit Analysis

This notebook follows the pedagogical flow of scqubits `demo_customcircuit.ipynb`, but applies it to the grounded Yan-style tunable-coupler circuit currently supported in `ScQubitsMimic.jl`.

The physical system is not the scqubits `0-π` circuit. Instead, we study a three-loop grounded circuit with left-qubit, coupler, and right-qubit SQUID loops. Supported symbolic and direct-diagonalization APIs are arranged in the same order as the scqubits tutorial, while unsupported sections are left in place as explicit notes.


## Circuit Information and Define Circuit

The YAML description below builds a grounded three-node circuit. The three superconducting loops will later be identified with `Φ1` (left qubit), `Φ2` (middle coupler), and `Φ3` (right qubit).


In [2]:
tcap_coupled_tmon = """
branches:
  - [JJ, 0, 1, EJ=4.5, EC=0.1]
  - [JJ, 1, 0, EJ=10.5, EC=0.1]
  - [C, 1, 0, EC=0.2]
  - [C, 1, 2, EC=5.0]
  - [JJ, 0, 2, EJ=30.0, EC=0.1]
  - [JJ, 2, 0, EJ=20.0, EC=0.1]
  - [C, 2, 0, EC=0.1]
  - [C, 2, 3, EC=5.0]
  - [JJ, 0, 3, EJ=4.6, EC=0.1]
  - [JJ, 3, 0, EJ=10.0, EC=0.1]
  - [C, 3, 0, EC=0.2]
"""

circ1 = Circuit(tcap_coupled_tmon; ncut=6)


UndefVarError: UndefVarError: `Circuit` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Lagrangian and Variable Transformation

We first inspect the symbolic Lagrangian in the node basis, then compare it with the transformed-variable basis used internally for classification and quantization.


**Lagrangian in terms of node variables**


In [3]:
sym_lagrangian(circ1; vars_type=:node)


UndefVarError: UndefVarError: `sym_lagrangian` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

**Transformation between node variables and new variables**


In [4]:
variable_transformation(circ1)


UndefVarError: UndefVarError: `variable_transformation` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [5]:
circ1.transformation_matrix


UndefVarError: UndefVarError: `circ1` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

**Types of the transformed variables**


In [6]:
circ1.var_categories


UndefVarError: UndefVarError: `circ1` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

**Lagrangian in terms of new variables**


In [7]:
sym_lagrangian(circ1; vars_type=:new)


UndefVarError: UndefVarError: `sym_lagrangian` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Hamiltonian and Diagonalization

In the current implementation this circuit is treated as three periodic modes, so the low-lying spectrum is obtained by direct diagonalization in a global charge-basis truncation.


**Symbolic Hamiltonian**


In [8]:
sym_hamiltonian_node(circ1)


UndefVarError: UndefVarError: `sym_hamiltonian_node` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
sym_hamiltonian(circ1)


UndefVarError: UndefVarError: `sym_hamiltonian` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

**Offset-charge variables and transformation**

`offset_charges(circ1)` lists the periodic transformed-mode offset-charge symbols used by the numerical charging Hamiltonian. `offset_charge_transformation(circ1)` now exposes the scqubits-style linear mapping from node-charge placeholders `q_n1`, `q_n2`, `q_n3` to those `ng` variables. For this grounded example the transformation matrix is the identity, so the mapping is trivial, but the same helper follows the actual transformed-mode indexing rule `inv(Tᵀ)[mode, :]` in general.


In [10]:
offset_charges(circ1)


UndefVarError: UndefVarError: `offset_charges` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
offset_charge_transformation(circ1)


UndefVarError: UndefVarError: `offset_charge_transformation` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
circ_ng = Circuit(tcap_coupled_tmon; ncut=6)
set_param!(circ_ng, :ng2, 0.1)
evals_ng = eigenvals(circ_ng; evals_count=4)

println("Representative periodic-mode offset charge: ng2 = ", get_param(circ_ng, :ng2))
println("ω01 at ng2 = 0.1: ", evals_ng[2] - evals_ng[1], " GHz")


UndefVarError: UndefVarError: `Circuit` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

**Cutoff properties**

The repo now exposes scqubits-style cutoff properties on `Circuit`. The available names depend on `var_categories`: periodic modes use `cutoff_n_i`, while extended modes use `cutoff_ext_i`. Since `circ1` has three periodic modes and no extended modes, only `cutoff_n_1`, `cutoff_n_2`, and `cutoff_n_3` appear below.


In [13]:
circ_cut = Circuit(tcap_coupled_tmon; ncut=6)

println("cutoff_names = ", circ_cut.cutoff_names)
println("Initial cutoff_n_2 = ", circ_cut.cutoff_n_2)
println("Initial Hilbert dimension = ", hilbertdim(circ_cut))

circ_cut.cutoff_n_2 = 4

println("Updated cutoff_n_2 = ", circ_cut.cutoff_n_2)
println("Internal stored dimension for mode 2 = ", circ_cut.cutoffs[2])
println("Updated Hilbert dimension = ", hilbertdim(circ_cut))


UndefVarError: UndefVarError: `Circuit` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

**Current scope in this section**

Per-mode cutoff editing is now available for top-level `Circuit`. This notebook still keeps a single global `ncut=6` for the main analysis object `circ1`, and uses `circ_cut` above only to demonstrate how the per-mode cutoff API changes the truncated Hilbert space.


In [14]:
evals0 = eigenvals(circ1; evals_count=4)
transitions0 = (
    ω01 = evals0[2] - evals0[1],
    ω12 = evals0[3] - evals0[2],
    ω23 = evals0[4] - evals0[3],
)

println("Eigenvalues: ", evals0)
println("Transitions: ", transitions0)


UndefVarError: UndefVarError: `eigenvals` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Set External Fluxes

The scqubits tutorial next identifies the external-flux symbols and then sweeps one of them. For this Yan-style circuit, `sym_external_fluxes(circ1)` shows that `Φ1` closes the left-qubit loop, `Φ2` the middle coupler loop, and `Φ3` the right-qubit loop.


In [15]:
sym_external_fluxes(circ1)


UndefVarError: UndefVarError: `sym_external_fluxes` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

**Coupler-flux sweep**

We now fix `Φ1 = Φ3 = 0` and sweep only `Φ2`, which tunes the middle SQUID loop. In the present circuit convention the sweep variable is a phase bias in radians, so `Φ2 = π` corresponds to the frustration point of the coupler SQUID.


In [16]:
circ_flux = Circuit(tcap_coupled_tmon; ncut=6)
set_param!(circ_flux, "Φ1", 0.0)
set_param!(circ_flux, "Φ3", 0.0)

φ2_vals = collect(range(-π, π; length=101))
flux_sweep = get_spectrum_vs_paramvals(circ_flux, "Φ2", φ2_vals; evals_count=4)

relative_levels = flux_sweep.eigenvalues .- flux_sweep.eigenvalues[:, 1]
transitions = hcat(
    flux_sweep.eigenvalues[:, 2] .- flux_sweep.eigenvalues[:, 1],
    flux_sweep.eigenvalues[:, 3] .- flux_sweep.eigenvalues[:, 2],
    flux_sweep.eigenvalues[:, 4] .- flux_sweep.eigenvalues[:, 3],
)

println("Computed ", length(φ2_vals), " sweep points.")
println("First relative spectrum row: ", relative_levels[1, :])


UndefVarError: UndefVarError: `Circuit` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Spectrum Plots


In [17]:
fig = Figure(size=(900, 380))

ax1 = Axis(fig[1, 1],
    xlabel="Φ2 (rad)",
    ylabel="Eₖ - E₀ (GHz)",
    title="Low-lying spectrum vs coupler flux")

for k in 1:size(relative_levels, 2)
    lines!(ax1, φ2_vals, relative_levels[:, k], label="E$(k-1) - E0")
end
axislegend(ax1, position=:rb)

ax2 = Axis(fig[1, 2],
    xlabel="Φ2 (rad)",
    ylabel="Transition frequency (GHz)",
    title="Transition frequencies")
lines!(ax2, φ2_vals, transitions[:, 1], label="ω01")
lines!(ax2, φ2_vals, transitions[:, 2], label="ω12")
lines!(ax2, φ2_vals, transitions[:, 3], label="ω23")
axislegend(ax2, position=:rb)

fig


UndefVarError: UndefVarError: `Figure` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Hierarchical Diagonalization and Subsystem Interactions

In scqubits, the next section configures subsystem groupings and inspects subsystem Hamiltonians and interaction terms. The current repo does not yet expose the corresponding tutorial APIs:

- `configure(...)`
- `sym_hamiltonian(subsystem_index=...)`
- `sym_interaction(...)`


## Visualization Capabilities

The scqubits example also shows potential-energy cross-sections and circuit wavefunctions. Those are not yet available for `Circuit` in this repo:

- `plot_potential`
- `Circuit` wavefunction plotting
